# Módulo 06a · MLlib — Clasificación de gravedad en incidentes viales

**Pregunta:** ¿Qué factores predicen si un incidente vial en Medellín resultará en solo daños, heridos o muertos?

**Fuente:** `silver.incidentes_geocodificados` (MEData 2014-2024)  
**Modelo:** `RandomForestClassifier` (100 árboles, profundidad 10)  
**Target:** `gravedad` — multiclase (Solo daños / Con heridos / Con muertos)

In [ ]:
import sys
sys.path.insert(0, "/workspace/src")

from shared.config import (
    crear_spark_session,
    TBL_SILVER_INCIDENTES,
    TBL_GOLD_ML_FATALIDAD_EVAL,
)
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

spark = crear_spark_session("Notebook-ML-Fatalidad")
spark.sparkContext.setLogLevel("WARN")
print("SparkSession lista.")

## 1. Exploración de datos

In [ ]:
df_raw = spark.table(TBL_SILVER_INCIDENTES)
print(f"Total registros en Silver: {df_raw.count():,}")
df_raw.printSchema()

In [ ]:
# Distribución de la variable objetivo
print("Distribución de gravedad (variable objetivo):")
df_raw.groupBy("gravedad").count().orderBy(F.desc("count")).show()

In [ ]:
# Distribución de clase de accidente
print("Distribución por clase de accidente:")
df_raw.groupBy("clase").count().orderBy(F.desc("count")).show()

In [ ]:
# Top 10 comunas con más incidentes
print("Top 10 comunas por incidentes:")
df_raw.groupBy("comuna").count().orderBy(F.desc("count")).show(10)

## 2. Preparación de features

In [ ]:
CLASES_VALIDAS = ("Con heridos", "Con muertos", "Solo daños")
COLS_CATEGORICAS = ["clase", "diseno_via", "comuna"]
COLS_NUMERICAS = ["hora", "dia_semana", "mes_accidente", "longitud", "latitud"]

df = (
    df_raw
    .filter(F.col("gravedad").isin(*CLASES_VALIDAS))
    .filter(F.col("fecha_accidente_ts").isNotNull())
    .withColumn("hora", F.hour("fecha_accidente_ts"))
    .withColumn("dia_semana", F.dayofweek("fecha_accidente_ts"))
    .fillna("DESCONOCIDO", subset=COLS_CATEGORICAS)
    .fillna(0.0, subset=["longitud", "latitud"])
)

print(f"Registros para ML: {df.count():,}")

In [ ]:
# Incidentes fatales por hora del día
print("Incidentes Con muertos por hora del día:")
(
    df.filter(F.col("gravedad") == "Con muertos")
      .groupBy("hora").count()
      .orderBy("hora")
      .show(24)
)

## 3. Pipeline MLlib

In [ ]:
SEMILLA = 42

df_train, df_test = df.randomSplit([0.8, 0.2], seed=SEMILLA)
print(f"Train: {df_train.count():,} | Test: {df_test.count():,}")

label_indexer = StringIndexer(inputCol="gravedad", outputCol="label", handleInvalid="keep")
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in COLS_CATEGORICAS
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in COLS_CATEGORICAS
]
feature_cols = [f"{c}_ohe" for c in COLS_CATEGORICAS] + COLS_NUMERICAS
assembler = VectorAssembler(
    inputCols=feature_cols, outputCol="features", handleInvalid="keep"
)
rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=100, maxDepth=10, seed=SEMILLA,
)

pipeline = Pipeline(stages=[label_indexer, *indexers, *encoders, assembler, rf])
print("Pipeline construido. Entrenando...")

In [ ]:
modelo = pipeline.fit(df_train)
print("Entrenamiento completado.")

## 4. Evaluación

In [ ]:
predicciones = modelo.transform(df_test)

for metrica_key, etiqueta in [
    ("accuracy",           "Accuracy"),
    ("f1",                 "F1 (weighted)"),
    ("weightedPrecision",  "Precision (weighted)"),
    ("weightedRecall",     "Recall (weighted)"),
]:
    ev = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName=metrica_key
    )
    print(f"{etiqueta:25s} {ev.evaluate(predicciones):.4f}")

In [ ]:
# Matriz de confusión (conteo predicción × real)
print("Matriz de confusión (gravedad real × predicción):")
predicciones.groupBy("gravedad", "prediction").count().orderBy("gravedad", "prediction").show()

## 5. Importancia de features

In [ ]:
rf_model = modelo.stages[-1]
importancias = rf_model.featureImportances.toArray()

# Reconstruir nombres de features expandidos
ohe_sizes = [
    modelo.stages[len(COLS_CATEGORICAS) + 1 + i].categorySizes[0] - 1
    for i in range(len(COLS_CATEGORICAS))
]

nombres_features = []
for col, sz in zip(COLS_CATEGORICAS, ohe_sizes):
    nombres_features += [f"{col}_{j}" for j in range(sz)]
nombres_features += COLS_NUMERICAS

pares = sorted(zip(importancias, nombres_features), reverse=True)
print("Top 15 features por importancia:")
for imp, nombre in pares[:15]:
    print(f"  {nombre:30s}  {imp:.4f}")

In [ ]:
# Métricas guardadas en Gold
print("Métricas en Gold:")
spark.table(TBL_GOLD_ML_FATALIDAD_EVAL).show(truncate=False)

## Conclusiones

- El modelo multiclase enfrenta **desbalance de clases**: `Solo daños` es mayoría, `Con muertos` es minoría.
- Las features más predictivas tienden a ser la **clase de accidente** (choque, atropello, volcamiento)
  y la **hora** del suceso (madrugada concentra fatalidades).
- Para mejorar el modelo en iteraciones futuras: aplicar **SMOTE** o pesos por clase
  (`weightCol` en RandomForest) para mejorar recall en la clase `Con muertos`.
- Las coordenadas (`longitud`, `latitud`) ofrecen señal geográfica complementaria a la `comuna`.